# Tutorial 1: Basic EEG Pipeline — Age Prediction with LightGBM

This tutorial walks through a complete EEG analysis pipeline using the **EEGDash** package: from loading open-access EEG datasets, through preprocessing and feature extraction, to training a machine learning model that predicts participant age from resting-state EEG.

**What you will learn:**
- How to load EEG recordings from EEGDash with metadata filtering
- How to preprocess EEG using braindecode's `Preprocessor` API (channel selection, average re-referencing, notch filtering, resampling, and bandpass filtering)
- How to extract spectral and connectivity features with `FeatureExtractor`
- Why subject-level train/test splitting is essential to avoid data leakage in windowed data
- How to fit and interpret a LightGBM regression model

**Required packages:** `eegdash`, `braindecode`, `lightgbm`, `shap`, `scikit-learn`, `pandas`, `numpy`, `matplotlib`

---
## 0. Exploring the Dataset

Before loading any data, it's worth getting a feel for what's actually in the two HBN releases. The `EEGDash` client gives you a lightweight way to query the metadata database directly — no files are downloaded, just JSON records from the server.

We'll check how many resting-state recordings are available, peek at a sample record, and look at the distribution of ages across subjects.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from eegdash import EEGDash

eeg = EEGDash()

# --- How many resting-state recordings exist in R1? ---
n_total = eeg.count({"dataset": "ds005505", "task": "RestingState"})
print(f"Resting-state recordings (ds005505 / R1): {n_total}")

# --- Peek at a single record to see what fields are available ---
sample = eeg.find_one(dataset="ds005505", task="RestingState")
print("\nTop-level fields:", [k for k in sample.keys() if not k.startswith("_")])
print("\nSubject metadata (participant_tsv):")
for k, v in sample["participant_tsv"].items():
    print(f"  {k}: {v}")

# --- Channel names available in the HBN recordings ---
ch_names = sample.get("ch_names", [])
print(f"\nNumber of channels : {len(ch_names)}")
print(f"All channel names  : {ch_names}")

# --- Fetch all R1 records and inspect the age distribution ---
# Age and sex are stored in the nested participant_tsv field, not at the top level
records = eeg.find({"dataset": "ds005505", "task": "RestingState"})
meta_df = pd.DataFrame(records)
meta_df["age"] = meta_df["participant_tsv"].apply(lambda x: x.get("age"))
meta_df["sex"] = meta_df["participant_tsv"].apply(lambda x: x.get("sex"))

print(f"\nTotal recordings : {len(meta_df)}")
print(
    f"Age range        : {meta_df['age'].min():.1f} – {meta_df['age'].max():.1f} years"
)
print(f"Sex breakdown    : {meta_df['sex'].value_counts().to_dict()}")

# Age distribution
fig, ax = plt.subplots(figsize=(7, 3))
meta_df["age"].astype(float).hist(bins=25, ax=ax, color="steelblue", edgecolor="white")
ax.set_xlabel("Age (years)")
ax.set_ylabel("Number of recordings")
ax.set_title("Age distribution — HBN resting-state (R1)")
plt.tight_layout()
plt.show()

---
## 1. Data Loading

EEGDash provides a unified interface to open-access EEG datasets stored in BIDS format. We load the HBN R1 release (`ds005505`) and filter to the `RestingState` task. Recordings are downloaded to a local cache on the first run — subsequent runs read directly from disk.

The `description_fields` argument controls which metadata columns are attached to each recording. Including `"age"` and `"sex"` makes them available per-recording later in the pipeline.

> **Memory requirements:** The full R1 dataset contains ~143 resting-state recordings (~130 MB each). Loading and preprocessing all of them keeps ~19 GB of raw data in RAM simultaneously, and FFT-based bandpass filtering adds further large temporary arrays. This tutorial therefore loads a configurable subset controlled by `N_PER_RELEASE`. To ensure the subset is representative, subjects are sampled **proportionally across age quartiles** so the training distribution mirrors the full cohort. To use the full dataset, set `N_PER_RELEASE = None` and remove the list-slicing in the loading cell below.

In [ ]:
import pandas as pd
from eegdash import EEGDash, EEGDashDataset

# ── Memory knob ───────────────────────────────────────────────────────────────
# 80 subjects ≈ 10 GB peak RAM.
# Set to None to load the full dataset.
N_SUBJECTS = 80


def select_age_balanced(records, n, seed=42):
    """Sample n records that mirror the full age distribution (quartile-proportional).

    Uses proportional allocation per age quartile. When Python's banker's rounding
    causes the per-bin counts to sum to fewer than n, the shortfall is filled from
    the remaining pool to guarantee exactly n records are returned.
    """
    df = pd.DataFrame(
        {
            "record": records,
            "age": [r.get("participant_tsv", {}).get("age") for r in records],
        }
    ).dropna(subset=["age"])
    df["age"] = df["age"].astype(float)
    if len(df) <= n:
        return df["record"].tolist()
    df["bin"] = pd.qcut(df["age"], q=4, labels=False, duplicates="drop")
    sampled = df.groupby("bin", group_keys=False).apply(
        lambda g: g.sample(n=max(1, round(n * len(g) / len(df))), random_state=seed),
        include_groups=False,
    )
    records_out = sampled["record"].tolist()
    # Banker's rounding on equal-sized bins can yield fewer than n (e.g. round(2.5)→2
    # for all four bins gives 8 instead of 10). Top up from the remaining pool.
    if len(records_out) < n:
        remaining = df[~df.index.isin(sampled.index)]
        top_up = remaining.sample(n=n - len(records_out), random_state=seed)
        records_out += top_up["record"].tolist()
    return records_out[:n]


# Fetch metadata only (no EEG files downloaded yet), then sample age-balanced subset
_client = EEGDash()
all_r1 = list(_client.find({"dataset": "ds005505", "task": "RestingState"}))

records_r1 = select_age_balanced(all_r1, N_SUBJECTS)

_shared = dict(
    description_fields=["subject", "session", "run", "task", "age", "sex"],
    cache_dir="./eegdash_data",
)

ds = EEGDashDataset(dataset="ds005505", records=records_r1, **_shared)  # R1

print(f"Total recordings : {len(ds.datasets)}")

# Individual dataset descriptions are populated lazily (on first raw access).
# The metadata is already available in the records we fetched from the API.
sample = records_r1[0]
print(f"\nSample — subject : {sample.get('subject')}")
print(f"         age     : {sample.get('participant_tsv', {}).get('age')}")
print(f"         task    : {sample.get('task')}")

In [ ]:
print(ds.description)

---
## 2. Preprocessing

Raw EEG contains artifacts, noise, and irrelevant channels that must be addressed before feature extraction. We apply six steps in a single `preprocess()` call, in the following order:

| # | Step | Rationale |
|---|------|-----------|
| 1 | **Channel selection** (`pick_channels`) | Reduces from 128+ EGI channels to 20 immediately, cutting memory and compute for all subsequent steps. |
| 2 | **Rename channels** (`rename_channels`) | Maps EGI HydroCel names (E22, E36…) to standard 10-20 labels (Fp1, C3…) so all downstream code is readable. |
| 3 | **Average re-reference** (`set_eeg_reference`) | Re-referencing should precede filtering. Filtering a signal and then re-referencing can subtly distort the spectral contribution of the reference channel. |
| 4 | **Notch filter at 60 Hz** (`notch_filter`) | Must run *before* resampling to 128 Hz (Nyquist = 64 Hz). Removes 60 Hz line noise and its harmonics (120 Hz, 180 Hz…) before they can alias into the kept frequency band during downsampling. |
| 5 | **Resample to 128 Hz** (`resample`) | MNE applies an anti-aliasing filter automatically. The 4× reduction in data size makes all later computations faster. |
| 6 | **Bandpass 1–55 Hz** (`filter`) | Final spectral shaping on the clean, downsampled signal. No aliasing risk since 55 Hz < 64 Hz Nyquist. Removes slow drift (<1 Hz) and high-frequency noise above the gamma band. |

> **Why `n_jobs=2`:** `preprocess()` with `n_jobs=-1` spawns one worker per CPU core, each loading its assigned recording into RAM simultaneously. Setting `n_jobs=2` caps that to two recordings in flight at once, avoiding a large transient memory spike. If your system has ≥ 64 GB RAM you can safely raise this value.

In [ ]:
from braindecode.preprocessing import Preprocessor, preprocess

# Maps standard 10-20 labels → EGI HydroCel GSN 128 channel names.
# Keys are the readable names used throughout the rest of the notebook;
# values are the raw EGI names present in the data files.
CH_NAMES = {
    "Fp1": "E22",
    "Fp2": "E9",
    "F7": "E33",
    "F3": "E24",
    "Fz": "E11",
    "F4": "E124",
    "F8": "E122",
    "T7": "E45",
    "C3": "E36",
    "Cz": "Cz",
    "C4": "E104",
    "T8": "E108",
    "P7": "E58",
    "P3": "E52",
    "Pz": "E62",
    "P4": "E92",
    "P8": "E96",
    "O1": "E70",
    "Oz": "E75",
    "O2": "E83",
}

preprocess(
    ds,
    [
        Preprocessor(
            "pick_channels", ch_names=list(CH_NAMES.values())
        ),  # 1. select 20 channels (EGI names)
        Preprocessor(
            "rename_channels", mapping={v: k for k, v in CH_NAMES.items()}
        ),  # 2. EGI → 10-20 names
        Preprocessor(
            "set_eeg_reference", ref_channels="average"
        ),  # 3. average re-reference
        Preprocessor("notch_filter", freqs=60),  # 4. remove 60 Hz line noise
        Preprocessor("resample", sfreq=128),  # 5. downsample to 128 Hz
        Preprocessor("filter", l_freq=1.0, h_freq=55.0),  # 6. bandpass 1–55 Hz
    ],
    n_jobs=2,
)

sample_raw = ds.datasets[0].raw
print(f"Channels      : {len(sample_raw.ch_names)}")
print(f"Sampling rate : {sample_raw.info['sfreq']} Hz")
print(f"Channels      : {sample_raw.ch_names}")

---
## 3. Windowing

HBN resting-state recordings follow a fixed protocol:

```
resting_start → [ instructed_toOpenEyes → 20 s → instructed_toCloseEyes → 40 s ] × N
```

Before windowing we apply `hbn_ec_ec_reannotation`, which converts the raw `instructed_toCloseEyes` BIDS markers into a series of `eyes_closed` annotations spaced 2 seconds apart from 15 s to 29 s after each instruction onset. This skips the first 15 seconds of each block (transition period) and keeps only the clean, steady eyes-closed signal.

`create_windows_from_events` with `mapping={"eyes_closed": 0}` then creates fixed-length windows starting at each `eyes_closed` event. Since the events are 2 s apart and each window is 4 s long, adjacent windows overlap by 50% — a well-accepted trade-off for feature-based ML that roughly doubles the number of training examples without inflating memory.

> **Expected output:** MNE INFO logs and a `RuntimeWarning: Omitted N annotation(s) that were outside data range` are suppressed below and are expected. They occur when an eyes-closed block falls near the end of a recording and some annotation windows extend past the available data — those windows are simply dropped.

In [ ]:
import warnings
import mne
from eegdash.hbn.preprocessing import hbn_ec_ec_reannotation
from braindecode.preprocessing import preprocess, create_windows_from_events

mne.set_log_level("WARNING")
warnings.filterwarnings("ignore", message="Omitted.*annotation")

preprocess(ds, [hbn_ec_ec_reannotation()])

sfreq = ds.datasets[0].raw.info["sfreq"]
window_size_samples = int(4 * sfreq)  # 4-second windows
window_stride_samples = int(2 * sfreq)  # 50% overlap

windows_ds = create_windows_from_events(
    ds,
    trial_start_offset_samples=0,
    trial_stop_offset_samples=window_size_samples,
    window_size_samples=window_size_samples,
    window_stride_samples=window_stride_samples,
    drop_last_window=True,
    mapping={"eyes_closed": 0},
)

total_windows = sum(len(d) for d in windows_ds.datasets)
print(f"Total windows (eyes-closed only): {total_windows}")
print(f"Windows in first recording: {len(windows_ds.datasets[0])}")

---
## 4. Feature Extraction

Raw EEG windows are high-dimensional and non-stationary. We summarise each window with three families of interpretable features, using a nested `FeatureExtractor` to keep them organised. Each level of the nesting can declare its own preprocessor, which runs **once** per batch regardless of how many features in that group depend on it.

**Non-alpha spectral features** (all channels, via Welch PSD):
- `spectral_bands_power` with custom `NON_ALPHA_BANDS` (delta/theta/beta) — alpha is **excluded** here because it is extracted separately from occipital channels below
- `spectral_entropy` — Shannon entropy of the normalised power spectrum
- `spectral_slope` — the 1/f exponent; flatter slopes are associated with ageing

**Occipital alpha** (occipital channels only):
- `pick_channels_preprocessor` (from the feature bank) selects only occipital electrodes before computing the PSD. A second nested level then runs `spectral_preprocessor` → `spectral_bands_power` with `ALPHA_BAND` only. This isolates the prominent eyes-closed alpha rhythm from the posterior scalp while leaving it out of the all-channel spectral features.

**Connectivity features** (all channel pairs):
- `connectivity_magnitude_square_coherence` (MSC) — frequency-resolved synchronisation between pairs. High inter-regional alpha coherence is a known marker of healthy ageing.

`partial()` is used throughout to pre-bind keyword arguments (`bands=`, `channels=`) that the `FeatureExtractor` cannot pass directly — the pipeline calls preprocessors and features with only the data and `_metadata`.

In [ ]:
from functools import partial
from eegdash.features import (
    FeatureExtractor,
    extract_features,
    spectral_preprocessor,
    spectral_normalized_preprocessor,
    spectral_db_preprocessor,
    spectral_bands_power,
    spectral_entropy,
    spectral_slope,
    spectral_edge,
    connectivity_coherency_preprocessor,
    connectivity_magnitude_square_coherence,
    connectivity_imaginary_coherence,
)
from eegdash.features.feature_bank.pick import pick_channels_preprocessor

# Alpha is extracted separately from occipital channels; exclude it from the all-channel branch
NON_ALPHA_BANDS = {"delta": (1, 4.5), "theta": (4.5, 8), "beta": (12, 30)}
ALPHA_BAND = {"alpha": (8, 12)}

# All three occipital electrodes are in the channel selection after renaming
OCCIPITAL_CHS = ["O1", "Oz", "O2"]

# Symmetric frontal–parietal connectivity pairs (standard 10-20 names post-rename)
# Fz–Pz : frontoparietal midline
# F3–P3 : left hemisphere
# F4–P4 : right hemisphere
CON_CHS_PAIRS = [("Fz", "Pz"), ("F3", "P3"), ("F4", "P4")]

extractor = FeatureExtractor(
    feature_extractors={
        "spectral": FeatureExtractor(
            preprocessor=spectral_preprocessor,
            feature_extractors={
                "occipital": FeatureExtractor(
                    preprocessor=partial(
                        pick_channels_preprocessor, channels=OCCIPITAL_CHS
                    ),
                    feature_extractors={
                        "alpha": partial(spectral_bands_power, bands=ALPHA_BAND)
                    },
                ),
                "bands": partial(spectral_bands_power, bands=NON_ALPHA_BANDS),
                0: FeatureExtractor(
                    preprocessor=spectral_normalized_preprocessor,
                    feature_extractors={
                        "entropy": spectral_entropy,
                        "edge": spectral_edge,
                    },
                ),
                1: FeatureExtractor(
                    preprocessor=spectral_db_preprocessor,
                    feature_extractors={
                        "slope": spectral_slope,
                    },
                ),
            },
        ),
        "connectivity": FeatureExtractor(
            preprocessor=partial(
                connectivity_coherency_preprocessor, pairs=CON_CHS_PAIRS
            ),
            feature_extractors={
                "msc": connectivity_magnitude_square_coherence,
                "icoh": connectivity_imaginary_coherence,
            },
        ),
    }
)

features_ds = extract_features(windows_ds, extractor, batch_size=512, n_jobs=-1)

## Inspect the feature output

In [ ]:
sample_feat_ds = features_ds.datasets[0]
feat_df = sample_feat_ds.features

print(f"Feature matrix shape (windows x features): {feat_df.shape}")

print("\nFeature families:")
for prefix in ["spectral", "connectivity"]:
    cols = [c for c in feat_df.columns if c.startswith(prefix)]
    print(f"  {prefix}: {len(cols)} columns")

print("\nAll feature column names:")
for col in feat_df.columns:
    print(f"  {col}")
print("  ...")

---
## 5. Data Preparation — Preventing Data Leakage

This step is critical and easy to get wrong. Because we created many overlapping windows from each recording, a naive random split of windows would place windows from the **same subject** in both train and test sets. The model would then "memorise" individual subjects' EEG signatures rather than learning generalisable patterns — a form of data leakage that inflates test performance.

**The correct approach:** split at the subject level so that all windows from a given subject appear exclusively in either train or test. We also reduce each subject to a single row by averaging features across their windows, which further removes any window-level correlation.

Each `FeaturesDataset` in `features_ds.datasets` corresponds to one recording. We read the subject ID and age from `.description` (recording-level metadata), then average the per-window features from `.features`.

In [ ]:
print(ds.description)

In [ ]:
import pandas as pd

rows = []
for feat_ds in features_ds.datasets:
    subj = feat_ds.description["subject"]
    age = feat_ds.description.get("age")
    if age is None:
        continue
    # Average features across all windows for this recording
    mean_feats = feat_ds.features.mean(axis=0)
    row = {"subject": subj, "age": float(age)}
    row.update(mean_feats.to_dict())
    rows.append(row)

subject_df = pd.DataFrame(rows).dropna()
print(f"Subjects with valid age labels: {len(subject_df)}")
print(f"Age range: {subject_df['age'].min():.0f} – {subject_df['age'].max():.0f} years")
subject_df[["subject", "age"]].head()

In [ ]:
from sklearn.model_selection import train_test_split

X = subject_df.drop(columns=["subject", "age"]).values
y = subject_df["age"].values

# Split at the subject level — each subject appears in only one split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training subjects: {X_train.shape[0]}")
print(f"Test subjects:     {X_test.shape[0]}")
print(f"Features per subject: {X_train.shape[1]}")

---
## 6. LightGBM Age Prediction

We use a gradient-boosted decision tree model (LightGBM) to predict age from the EEG features. LightGBM handles high-dimensional tabular data well without requiring feature scaling and natively handles feature importance. We report **Mean Absolute Error (MAE)** — the average error in years — and **R²** — the proportion of age variance explained by the model.

> **Note:** These two datasets together contain a relatively small number of subjects. The metrics below are intended to illustrate the pipeline, not to represent publication-quality predictive performance. Larger cohorts and additional preprocessing (e.g., artefact rejection, ICA) would be needed for a rigorous study.

In [ ]:
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, r2_score

model = LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Test MAE: {mae:.2f} years")
print(f"Test R²:  {r2:.3f}")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, y_pred, alpha=0.7, edgecolors="k", linewidths=0.5, s=60)

# Perfect prediction line
lims = [min(y_test.min(), y_pred.min()) - 2, max(y_test.max(), y_pred.max()) + 2]
ax.plot(lims, lims, "--", color="gray", linewidth=1.5, label="Perfect prediction")

ax.set_xlabel("Actual age (years)", fontsize=12)
ax.set_ylabel("Predicted age (years)", fontsize=12)
ax.set_title(f"Age Prediction: MAE = {mae:.2f} yr,  R² = {r2:.3f}", fontsize=12)
ax.legend(fontsize=10)
ax.set_xlim(lims)
ax.set_ylim(lims)
plt.tight_layout()
plt.show()

---
## 7. Extension: SHAP Feature Importance (Optional)

SHAP (SHapley Additive exPlanations) attributes each prediction to individual features in a theoretically principled way. The beeswarm plot below shows which EEG features drive age predictions: each dot is one test subject, coloured by feature value, positioned horizontally by its SHAP contribution.

This step requires the `shap` package (`pip install shap`) and adds significant computation time on large feature sets. It is optional — the rest of the tutorial is fully self-contained without it.

In [ ]:
# Optional — requires: pip install shap
try:
    import shap

    feature_names = subject_df.drop(columns=["subject", "age"]).columns.tolist()

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test)

    plt.figure()
    shap.summary_plot(
        shap_values,
        X_test,
        feature_names=feature_names,
        plot_type="dot",
        max_display=20,
        show=True,
    )
except ImportError:
    print("shap is not installed. Run: pip install shap")

---
## Summary

In this tutorial you built a complete EEG-to-prediction pipeline:

1. **Loaded** resting-state EEG recordings from the HBN R1 dataset via `EEGDashDataset`
2. **Preprocessed** the data in five ordered steps: channel selection (20 standard 10-20), average re-referencing, notch filtering at 60 Hz, resampling to 128 Hz, and bandpass filtering at 1–55 Hz
3. **Windowed** continuous recordings into 4-second eyes-closed epochs with 50% overlap
4. **Extracted** spectral band power and inter-channel coherence features
5. **Aggregated** features at the subject level and split correctly to avoid leakage
6. **Trained** a LightGBM regressor and evaluated it with MAE and R²
7. **Interpreted** predictions with SHAP

**Next steps:** Try Tutorial 2 for classification tasks, or explore `eegdash.features` for additional feature types such as Hjorth parameters and non-linear complexity measures.